# Transaction Intelligence — Phase 5: distillation (Colab)

**Before you run:** **Runtime → Change runtime type → GPU**; if you ran cells earlier, **Runtime → Restart session**. Then **Run all**.

Self-contained: trains the DistilBERT teacher, distills a tiny `bert-tiny` student, and trains the same student **from scratch** (ablation). The headline number is the **distillation lift** = `student` − `student-scratch`. Bar to beat (baseline): gold subtype 0.77 / category 0.85.

In [ ]:
# 1) Get the code (idempotent).
%cd /content
!rm -rf transaction-intelligence
!git clone https://github.com/thejayvaghela/transaction-intelligence.git
%cd transaction-intelligence

In [ ]:
# 2) Install deps WITHOUT touching Colab's PyTorch.
!pip install -q "transformers>=4.41" datasets accelerate sentencepiece mlflow-skinny
!pip install -q -e . --no-deps

In [ ]:
# 3) Rebuild the dataset deterministically.
!python scripts/build_dataset.py

In [ ]:
# 4) Train the DistilBERT teacher (~2 min) -> models/teacher (needed for distillation).
!python scripts/train_transformer.py

In [ ]:
# 5) Distill the teacher -> tiny bert-tiny student -> models/student. Prints student/test + student/gold.
!python scripts/distill.py

In [ ]:
# 6) Ablation: same bert-tiny trained FROM SCRATCH (hard labels only) -> models/student-scratch.
#    distillation lift = student macro-F1  -  student-scratch macro-F1.
!python scripts/train_transformer.py --model prajjwal1/bert-tiny --output models/student-scratch

In [ ]:
# 7) (Optional) Download all models.
!zip -rq models.zip models
from google.colab import files
files.download('models.zip')

## After running
Paste these back to Claude:
- **`student/test`** + **`student/gold`** (distilled bert-tiny)
- **`student-scratch/test`** + **`student-scratch/gold`** (same model, no distillation)
- (cell 4 also printed **`teacher/*`** for reference)

The **distillation lift** (student − student-scratch) is the headline. We also compare the tiny student to the teacher (~15× larger) and the baseline (gold 0.77 / 0.85), then move to Phase 6 (quantization).